# 2. Idealista · 02 · Listados de Búsqueda por Provincia
**Propósito**: Extrae datos resumidos de todas las ofertas de los municipios de la provincia (precio, título, descripción...). Hay que aclarar que esta información es resumida por que lo que hace el trabajo es escrapear las diferentes páginas de anuncios inmobiliarios de cada municipio, en estás páginas tenemos información de cada anuncio pero la información no es tan completa como la que aparece dentro de cada uno de los anuncios, el propósito es recuperar esta información, y en particular el enlace de cada anuncio para recuperar toda la información disponible de cada anuncio.

In [1]:
import sys
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from datetime import datetime
import asyncio
import pandas as pd

sys.path.insert(0, '/tfm/python_notebooks')
from tfm_lib import idealista_scrap as idealista
from tfm_lib import utils as tfm_utils

## Parámetros

In [ ]:
debug_mode = False  # True: entorno TEST gratis (1 registro)
provincia = 'murcia'

# Obtener sesión Spark (reutiliza si ya existe)
spark = tfm_utils.get_spark_session(f'Idealista_Listados_{provincia}')

## Funciones
Se definen las funciones necesarias para llevar a cabo el escrapero de cada municipio, se guardan los datos y se actualiza en la tabla original para indicar que ya ha sido escrapeado dicho municipio, de esta forma no escrapeamos repetidamente el mismo municipio, independientemente de las ejecuciones que se hagan sobre este script.

In [3]:
async def scrape_todos_los_municipios(urls_municipios: list, total_pages_list: list = None):
    """
    Scrapea todas las ofertas de una lista de municipios en paralelo.
    total_pages_list: lista con el número de páginas de cada municipio (campo total_paginas
    de la tabla de municipios). Si se pasa, se usa como referencia fiable en lugar de
    intentar detectar las páginas desde el paginador HTML de Idealista.
    """
    if total_pages_list:
        tasks = [idealista.scrape_search_listings(url, known_total_pages=tp, debug=debug_mode)
                 for url, tp in zip(urls_municipios, total_pages_list)]
    else:
        tasks = [idealista.scrape_search_listings(url, debug=debug_mode) for url in urls_municipios]
    resultados = await asyncio.gather(*tasks)
    ofertas=[oferta for municipio_res in resultados for oferta in municipio_res]
    return ofertas

## Leer Municipios de la Provincia
Leemos solo la partición de esta provincia desde la tabla de municipios.

In [ ]:
municipios_table="raw.idealista.municipios_urls"
df_municipios = (tfm_utils.read_from_delta(spark, municipios_table)
                    .filter(F.col("provincia")==provincia)
                    #Leesmos solo los no procesados
                    .filter(F.col("_scraped")==0)
                )

print(f'{df_municipios.count()} municipios a procesar en {provincia}')
print(datetime.now())

1 municipios a procesar en murcia
2026-04-28 20:41:07.916089


In [5]:
tfm_utils.display(df_municipios,5)

,municipio,municipio_url,num_ofertas,provincia,total_paginas,_scraped
0,murcia,https://www.idealista.com/venta-viviendas/murc...,3547,murcia,119,1


## Proceamos el escrapeo
Recorremos los diferentes municipios escrapeando sus ofertas uno a uno.

In [6]:
municipios_list=df_municipios.collect()

municipios_scrapeados=[]
municipios_no_scrapeados=[]

for municipio_row in municipios_list:
    municipio=municipio_row["municipio"]
    municipio_url=municipio_row["municipio_url"]
    municipio_num_pag=municipio_row["total_paginas"]

    try:
        #Scrapeamos todas las ofertas por municipio
        ofertas_municipio = idealista.run_async(scrape_todos_los_municipios([municipio_url], [municipio_num_pag]))
        
        #Leemos con Pandas para evitar el problema de inferencia de esquema
        #en Pandas si no infiere el tipo de datos de un campo lo pone como object, en PySpark fallaría
        df_pd=pd.DataFrame(ofertas_municipio).astype(str)
        df_pd["provincia"]=provincia
        df_pd["municipio"]=municipio_row["municipio"]
        
        df_pd=tfm_utils.normalize_df_pandas(df_pd)
        df_spark = spark.createDataFrame(df_pd)

        ofertas_table="raw.idealista.municipios_ofertas"
        ofertas_path=tfm_utils.full_table_path(ofertas_table)

        #Sobrescribimos por provincia y municipio
        (df_spark
            .write
            .partitionBy("provincia","municipio")
            .option("mergeSchema","true")
            .mode("overwrite")
            .format("delta")
            .save(ofertas_path)
        )
        
        print(f"Se han recuperado {df_spark.count()} ofertas para {municipio} en {provincia}")
        municipios_scrapeados.append(municipio)
        print(datetime.now())
        
    except Exception as e:
        print("Error:", e)
        traceback.print_exc()
        #Guardamos aquellos municipios que no han podido ser procesados por algún error en la función de escrapeo,
        #de esta forma aprovechamos las lecturas correctas y no gastamos tokens re-escrapeando dichos municipios
        municipios_no_scrapeados.append(municipio)

2026-04-28 20:41:10.540 | INFO     | tfm_lib.idealista_scrap.scraper:scrape_search_listings:229 - Scrapeando primera página de: https://www.idealista.com/venta-viviendas/murcia-murcia/
2026-04-28 20:42:02.615 | DEBUG    | tfm_lib.idealista_scrap.parser:parse_search_page_count:108 - Paginas calculadas desde h1 (3553 inmuebles): 119
2026-04-28 20:42:02.630 | INFO     | tfm_lib.idealista_scrap.scraper:scrape_search_listings:238 - Páginas obtenidas desde tabla de municipios para https://www.idealista.com/venta-viviendas/murcia-murcia/: 119
2026-04-28 20:42:02.630 | INFO     | tfm_lib.idealista_scrap.scraper:scrape_search_listings:250 - Municipio con 119 páginas > 72 — dividiendo por tipo de propiedad (pisos / casas-chalets)
2026-04-28 20:42:02.631 | INFO     | tfm_lib.idealista_scrap.scraper:_scrape_by_property_type:160 - Scrapeando subtype 'con-pisos' para murcia-murcia: https://www.idealista.com/venta-viviendas/murcia-murcia/con-pisos/
2026-04-28 20:42:02.633 | INFO     | tfm_lib.idealis

Se han recuperado 3321 ofertas para murcia en murcia
2026-04-28 20:52:02.094181


In [7]:
#Actualizamos los municipios escrapeados
if len(municipios_scrapeados)>0:
    #Cargamos todos los datos de los municipios de la provincia
    df_mun_full=(tfm_utils.read_from_delta(spark, municipios_table)
                    .filter(F.col("provincia")==provincia)
                     #Actualizamos el metadato de scrapeo
                     .withColumn("_scraped", F.when(F.col("_scraped")==1, F.col("_scraped"))
                                               .when(F.col("municipio").isin(municipios_scrapeados), F.lit(1))
                                               .otherwise(F.lit(0))
                                )
                )

    #Ahora tenemos el metadato actualizado, falta guardar esto en la tabla de municipios
    table_name="raw.idealista.municipios_urls"
    table_path=tfm_utils.full_table_path(table_name)

    (df_mun_full
        .write
        .partitionBy("provincia")
        .option("mergeSchema","true")
        .mode("overwrite")
        .format("delta")
        .save(table_path)
    )
    
    print(f"Dattos actualizados en {table_path}.\nConsultar en {table_name}.")
    print(datetime.now())

Dattos actualizados en /tfm/delta_lake/raw/idealista/municipios_urls.
Consultar en raw.idealista.municipios_urls.
2026-04-28 20:52:06.028594


In [8]:
print(municipios_scrapeados)
print(datetime.now())

['murcia']
2026-04-28 20:52:06.035432


In [9]:
print(municipios_no_scrapeados)
print(datetime.now())

[]
2026-04-28 20:52:06.046043


In [10]:
#En caso de que el proceso fallase con algún municipio marcamos el proceso completo para relanzar
#La siguiente ocasión se ejecutará solo sobre aquellos que no hayan sido escrapeados correctamente
if len(municipios_no_scrapeados)>0:
    raise Exception(f"Ha ocurrido un error al intentar escrapear la información de {municipios_no_scrapeados}")

print(datetime.now())

2026-04-28 20:52:06.055576
